In [103]:
import pandas as pd
import numpy as np
import random
import math

In [104]:
recipes_df = pd.read_csv("./data/recipes.csv")

def getmeals(recipes_df):
    meals = {}
    for _, row in recipes_df.iterrows():
       meals.update({ 
          row["Name"]:(
          row["type"],
          row["Category"],
          row["total_price"],
          row["provided_calories"]
       )})
    return meals

def gettransitionModel(recipes_df):
    result = {}

    for _, row in recipes_df.iterrows():
        meal_type = row["type"]
        name = row["Name"]

        if meal_type not in result:
            result[meal_type] = []

        result[meal_type].append(name)

    return result

def diversity_penalty_per_week(state):
    penalty = 0
    days_per_week = 7
    meals_per_day = 3

    for start in range(0, len(state), days_per_week):
        week = state[start:start + days_per_week]

        week_meals = [meal for day in week for meal in day]

        counts = {}
        for meal in week_meals:
            counts[meal] = counts.get(meal, 0) + 1

        for meal, count in counts.items():
            if count > 2:
                penalty += (count - 2)
    return penalty / (len(state) * meals_per_day)

class GAProblem:
    def __init__(self, recipes_df, total_price, tdee):
        self.meals = getmeals(recipes_df)
        self.size = 90
        self.total_price = total_price
        self.tdee = tdee
        self.transitionmodel = gettransitionModel(recipes_df)
    
    def get_total_cost(self, state):
        total_cost = 0
        for i in range(len(state)):
           breakfast_cost =  self.meals[state[i][0]][2]
           lunch_cost = self.meals[state[i][1]][2]
           dinner_cost  = self.meals[state[i][2]][2]
           day_cost = breakfast_cost+lunch_cost+dinner_cost
           total_cost += day_cost
        return total_cost

    def get_delta_nutritional(self, state):
       delta_dict = {}
       for i in range(len(state)):
           breakfast_c =  self.meals[state[i][0]][3]
           lunch_c = self.meals[state[i][1]][3]
           dinner_c  = self.meals[state[i][2]][3]
           day_c = breakfast_c+lunch_c+dinner_c
           delta_dict[state[i]] = abs(day_c-self.tdee)
       return delta_dict
    def get_total_cal(self,state):
        day_calories = []
        d = 0
        for day in state:
            day_calories.append(
                self.meals[day[0]][3] +
                self.meals[day[1]][3] +
                self.meals[day[2]][3]
            )
        for day in day_calories:
            if day > self.tdee *1.1 or day < self.tdee*0.9:
                if day:
                    print(day)
                print("Out of calories range")
        for i in range(len(day_calories)):
            d += day_calories[i]
        return d
   
    def fitness_function(self, state):
        total_cost = self.get_total_cost(state)
        cost_penalty = max(0, ((total_cost - self.total_price) / total_cost))
        lower_bound = 0.9 * self.tdee
        upper_bound = 1.1 * self.tdee

        bad_days = 0
        for day in state:
            day_calories = (
                self.meals[day[0]][3] +
                self.meals[day[1]][3] +
                self.meals[day[2]][3]
            )
            if day_calories < lower_bound or day_calories > upper_bound:
                bad_days += 1
        nutrition_penalty = bad_days / len(state)
        diversity_penalty = diversity_penalty_per_week(state)
        fitness = -(
            0.4 * cost_penalty +
            0.45 * nutrition_penalty +
            0.15 * diversity_penalty
        )
        return fitness
    def generate_random_state(self):
         return [(random.choice(self.transitionmodel['Breakfast']),
                  random.choice(self.transitionmodel['Lunch']),
                  random.choice(self.transitionmodel['Dinner']))
                    for _ in range(30)]
def crossover_and_mutation(population,problem):
       cutoff = random.randrange(1,29)
       childs = []
       for i in range(len(population)):
         for j in range(i+1,len(population)):
            parent_a = population[i][0]
            parent_b = population[j][0]
            child_a = parent_a[:cutoff] + parent_b[cutoff:]
            child_b = parent_b[:cutoff] + parent_a[cutoff:]
            random_lunch_a = random.choice(problem.transitionmodel['Lunch'])
            random_lunch_b = random.choice(problem.transitionmodel['Lunch'])
            random_pos_a  = random.randrange(30)
            random_pos_b  = random.randrange(30)
            t1 =   [child_a[random_pos_a][0],random_lunch_a, child_a[random_pos_a][2]]
            t2 =  [child_b[random_pos_b][0],random_lunch_b, child_b[random_pos_b][2]]
            child_a[random_pos_a] = tuple(t1) 
            child_a[random_pos_b] = tuple(t2) 
            childs.append((child_a,problem.fitness_function(child_a)))
            childs.append((child_b,problem.fitness_function(child_a)))
       return childs


def GASearch(problem):
    # step 1 Selection:
    list_of_states = []
    for _ in range(100):
        state = problem.generate_random_state()
        fitness = problem.fitness_function(state)
        list_of_states.append((state,fitness))
    population = sorted(list_of_states, key=lambda x: x[1], reverse=True)[:25]
    # step 2 and 3 crossover and mutation
    for _ in range(1000):
        population = sorted(population, key=lambda x: x[1], reverse=True)[:25]
        for state, fitness in population:
            if fitness == 0:
                return state
        population =  crossover_and_mutation(population,problem)
    best_candidate = sorted(population, key=lambda x: x[1], reverse=True)[:1]
    return best_candidate

  

In [105]:
g = GAProblem(recipes_df,22000,2000)
solution  = GASearch(g)
print(solution)
print(g.get_total_cost(solution[0][0]))
print(g.get_total_cal(solution[0][0]))

[([('Lentil Tomato Breakfast Soup', 'Chicken Olive Lemon Salad', 'Couscous Shrimp Dinner'), ('Lentil Tomato Breakfast Soup', 'Sardine Harissa Salad', 'Osbane'), ('Oat Honey Cinnamon Porridge', 'Couscous Beef Harissa', 'Beef Stir Fry'), ('Egg Oat Protein Pancakes', 'Lamb Zucchini Pepper Bowl', 'Chorba Vermicelle Beef'), ('Vegetable Chakchouka', 'Sardine Rice Tomato Bowl', 'Hchaichi'), ('Potato Onion Omelette DZ', 'Shrimp Rice Bowl', 'Tlitli Chicken Dinner'), ('Beef Egg Scramble', 'Rice & Egg Bowl', 'Chicken Pepper Skillet'), ('322 Bowl', 'Chickpea Chicken Bowl', 'Chicken Harira Style Bowl'), ('Tomato Pepper Omelette', 'Lamb Vegetable Couscous', 'Shrimp Pepper Plate'), ('Protein Smooth Bowl', 'Spinach Lentil Bowl', 'Harira Vegetarienne'), ('Algerian Chakchouka Eggs', 'Chorba Frik', 'Lamb Zucchini Dinner'), ('Mint Yogurt Honey Bowl', 'Tajine Zitoun', 'Vegetable Couscous'), ('Sardine Breakfast Plate', 'Tuna Rice Olive Bowl', 'Chorba Frik Style Chicken'), ('Protein Yogurt Crunch', 'Rice & E